In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
from pprint import pprint
import csv
import numpy as np

In [0]:
def get_json(url: str) -> dict:
    r = requests.get(url)
    r.raise_for_status()
    
    return r.json()

In [0]:
def fetch_all(first_url: str) -> list:
    url = first_url
    content = []
    
    while url:
        response = get_json(url)
        content.extend(response['data'])
        url = urljoin(first_url, response['next']) if response['next'] else None

    return content


orders = fetch_all(BASE_URL + 'orders/page-1.json')

In [0]:
def save_json(records: list, path:str) -> None:
    with open(path, 'w') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


save_json(orders, os.path.join(RAW_DIR, 'orders.json'))

In [0]:
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes,unchanged
    return len(response.content)

In [0]:
def extract(raw_dir: str) -> dict:
     orders = fetch_all(BASE_URL + 'orders/page-1.json')
     products = download_file(FILES_URL + 'products.csv', os.path.join(raw_dir,
'products.csv'))
     teams = download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir,
'teams.csv'))
     matches = download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir,
'matches.parquet'))
     
     df_orders = pd.json_normalize(orders, sep='_').rename(columns={'customer_name': 'customer', 'customer_city': 'city'})
     df_products = pd.read_csv(os.path.join(raw_dir,
'products.csv'))
     df_teams = pd.read_csv(os.path.join(raw_dir,
'teams.csv'))
     df_matches = pd.read_parquet(os.path.join(raw_dir,
'matches.parquet'))
     
     return {'orders': df_orders, 'products': df_products, 'teams': df_teams, 'matches': df_matches}

In [0]:
def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df_products = df.copy()
    duplicates_mask = df_products.duplicated('product_id', keep='first')
    df_products = df_products[~duplicates_mask]
    df_products['team_code'] = df_products['team_code'].str.strip().str.upper()
    df_products['category'] = df_products['category'].str.strip().str.lower()
    df_products['price'] = df_products['price'].str.replace(' RSD', '', regex=False)
    df_products['price'] = pd.to_numeric(df_products['price'], errors='coerce')
    df_products['active'] = np.where(df_products['active'].str.upper() == 'YES', True, False)

    return df_products

In [0]:
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df_orders = df.copy()
    for col in ['order_id', 'customer', 'city', 'payment']:
        df_orders[col] = df_orders[col].str.strip()
    df_orders['order_id'] = df_orders['order_id'].str.upper()
    df_orders['customer'] = df_orders['customer'].str.title()
    df_orders['city'] = df_orders['city'].str.title().fillna('Unknown')

    return df_orders

In [0]:
def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    duplicates_mask = df.duplicated('order_id', keep='first')
    return df[~duplicates_mask], df[duplicates_mask]

In [0]:
def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')
    price = pd.to_numeric(row['unit_price'], errors='coerce')
    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity != int(quantity) or quantity <= 0:
        return 'invalid_quantity'
    if pd.isna(price) or price <= 0:
        return 'invalid_price' 
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    
    return 'OK'

In [0]:
def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df_orders = df.copy()
    df_orders['quantity'] = pd.to_numeric(df_orders['quantity'], errors='coerce').astype(int)
    df_orders['unit_price'] = pd.to_numeric(df_orders['unit_price'], errors='coerce').astype(float)
    df_orders['order_date'] = pd.to_datetime(df_orders['order_date'], errors='coerce')
    df_orders['total'] = df_orders['quantity'] * df_orders['unit_price']

    prod_columns = products[['product_name', 'team_code', 'category', 'product_id']]
    
    return df_orders.merge(prod_columns, on='product_id', how='left')

In [0]:
def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df_raw = df.copy()
    df_cleaned_orders = clean_orders(df_raw)
    unique, duplicates = remove_duplicate_orders(df_cleaned_orders)
    duplicates = duplicates.assign(reason = 'duplicate')
    product_ids = set(products['product_id'])
    reasons = unique.apply(lambda row: validate_order(row, product_ids), axis=1)
    valid = transform_orders(unique[reasons == 'OK'], products)
    rejected = pd.concat([unique[reasons != 'OK'].assign(reason=reasons[reasons != 'OK']), duplicates])
    
    return valid, rejected
    

In [0]:
def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)
    if name.endswith('.parquet'):
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
    written[name] = len(df)
    
    return written

In [0]:
def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    outputs = {
    'clean_orders.parquet': valid,
    'rejected_orders.csv': rejected,
    'products.csv': products,
    'league_table.csv': league_table(results),
    'revenue_by_team.csv': revenue_by(valid.dropna(subset=
    ['team_code']), 'team_code'),
    'revenue_by_category.csv': revenue_by(valid, 'category'),
    'sales_after_results.csv': sales_after_results(valid, windows),
    }
    written = load(outputs, processed_dir)
    summary = {
    'total_rows': len(data['orders']),
    'duplicates': int((rejected['reason'] == 'duplicate').sum()),
    'valid_rows': len(valid),
    'rejected_rows': len(rejected),
    'total_revenue': round(float(valid['total'].sum()), 2),
    'files_written': len(written) + 1,
    }
    pd.DataFrame(list(summary.items()), columns=['metric',
    'value']).to_csv(
    os.path.join(processed_dir, 'summary.csv'), index=False)
    return summary